# PrivateLocalAgent · 可视化（无隧道）

界面直接嵌在 JupyterLab 里，**不需要 rc-tunnel / proxy / 公网端口**。

运行顺序：单元格 1 → 等到 `ready` → 单元格 2 出现可视化面板。

In [ ]:
# 1) 依赖 + 可选生成 FAQ + 初始化 Agent
import os, sys, subprocess, shutil
from pathlib import Path

def log(msg):
    print(msg, flush=True)

ROOT = Path("/workspace/Radeon-hackathon-2026-07")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
persist = Path("/workspace/persistence")
if persist.is_dir():
    os.environ.setdefault("PLA_DATA_ROOT", str(persist / "PrivateLocalAgent"))
    os.environ.setdefault("HF_HOME", str(persist / "huggingface"))
    Path(os.environ["PLA_DATA_ROOT"]).mkdir(parents=True, exist_ok=True)
    Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("USE_ROCM_AITER_ROPE_BACKEND", "0")

BUILD_10K = False  # 需要 1 万 FAQ 时改 True（首次很慢）
FORCE_REBUILD_KB = False

log("[0] pip...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "chromadb", "sentence-transformers", "pypdf", "pyyaml",
    "python-dotenv", "pydantic", "openai", "ipywidgets",
    "transformers", "accelerate", "safetensors", "sentencepiece",
])

if BUILD_10K:
    log("[1] generate 10k FAQs...")
    subprocess.check_call([sys.executable, "scripts/generate_kb_faqs.py", "--n", "10000"])

from src.agent.agent import PrivateAgent
from src.agent.planner import build_task_plan
from src.agent.tools import ToolRegistry
from src.config import load_settings
from src.llm.backend import build_llm
from src.memory.memory import SessionMemory
from src.rag.store import VectorStore

settings = load_settings()
persist = settings.resolve(settings.rag.persist_dir)
if FORCE_REBUILD_KB and persist.exists():
    shutil.rmtree(persist)

log("[2] KB...")
store = VectorStore(settings)
if store.count() == 0:
    n = store.add_directory(settings.resolve(settings.paths.sample_docs))
    log(f"    ingested: {n}")
else:
    log(f"    KB chunks: {store.count()}")

memory = SessionMemory(settings.resolve(settings.agent.memory_path))
tools = ToolRegistry(store, memory, settings.resolve(settings.paths.upload_dir))
log("[3] load LLM...")
llm = build_llm(settings.llm)
agent = PrivateAgent(llm, tools, memory, settings.agent.max_steps)
log("ready")

In [ ]:
# 2) 可视化面板（嵌在 Notebook 内，无隧道）
from src.app.notebook_visual import launch_notebook_visual
from src.agent.planner import build_task_plan

assert "agent" in globals(), "请先运行上一个单元格直到 ready"
ui = launch_notebook_visual(agent, plan_fn=build_task_plan)
# 若按钮界面失败，可改用： ui.ask("VPN 密码怎么重置？")

In [ ]:
# 3) 备用：无 widgets 时用这行（同样有可视化气泡）
# ui.ask("涉密文档可以用哪些 AI 工具？")